# Experiment: Qwen2.5-7B induction-copy mechanism surface (Phase 09)

**Question.** Does the preregistered Qwen2.5-7B induction-copy task pass its clean, discovery, and held-out confirmation gates, and can additive or quadratic observers recover its frozen eight-head intervention surface under fixed measurement budgets?

**Success criteria.** Use only the frozen configuration and stage order below. Treat a failed gate as a reportable negative stopping result. Do not make a positive response-surface or action claim until the locked-test stage finishes and the evaluation stage records the preregistered metrics. The 0.5B smoke run is engineering-only.

Experiments designed/concieved by Vijay Erramilli. Code written by Vijay Erramilli and Codex


## Before running

- Use a Colab A100, H100, or H200 runtime with at least 40 GB of GPU memory for the scientific 7B run. High host RAM helps with checkpoints but does not replace GPU memory.
- Put an already-authorized private ObserverBench checkout at `REPO_ROOT`, either by uploading it or mounting it from Drive. This notebook intentionally contains no repository URL, access token, or clone command.
- Record a private commit or tag for this source-sealed checkout before the run. The runner checks every producer source hash against the frozen manifest.
- Keep `ARTIFACTS_ROOT` on Google Drive. Every stage receives `--resume`; it resumes only when source, model, hardware, and dependency identity still match. If Colab changes that identity, start a fresh artifact root instead of mixing shards.
- Do not skip stages. In particular, observers and actions must be frozen before the locked-test outcomes are measured. The runner enforces prerequisites and exits nonzero when a scientific gate fails; that exception stops notebook execution.


In [ ]:
# Mount Drive when this notebook runs in Colab.
from pathlib import Path
import os

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

default_repo = Path("/content/ObserverBench") if IN_COLAB else Path.cwd()
REPO_ROOT = Path(os.environ.get("OBSERVERBENCH_REPO", default_repo)).expanduser().resolve()
default_artifacts = (
    Path("/content/drive/MyDrive/ObserverBenchArtifacts/phase09/qwen_induction/copy_v1")
    if IN_COLAB
    else REPO_ROOT / "results/revision/qwen_induction/copy_v1"
)
ARTIFACTS_ROOT = Path(os.environ.get("OBSERVERBENCH_PHASE09_ARTIFACTS", default_artifacts)).expanduser().resolve()
CONFIG = REPO_ROOT / "configs/revision/phase09/qwen2_5_7b_induction_full.json"
SOURCE_MANIFEST = REPO_ROOT / "configs/revision/phase09/qwen_phase09_source_manifest.json"
SCRIPT = REPO_ROOT / "scripts/run_qwen_induction_phase09.py"

assert (REPO_ROOT / "pyproject.toml").is_file(), f"Set REPO_ROOT to the private checkout: {REPO_ROOT}"
assert CONFIG.is_file(), f"Missing frozen configuration: {CONFIG}"
assert SOURCE_MANIFEST.is_file(), f"Missing frozen source manifest: {SOURCE_MANIFEST}"
assert SCRIPT.is_file(), f"Missing staged runner: {SCRIPT}"
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
{"repo_root": str(REPO_ROOT), "artifacts_root": str(ARTIFACTS_ROOT), "in_colab": IN_COLAB}


## Install the frozen experiment dependencies

The editable install keeps the notebook thin: prompt construction, interventions, checkpointing, gates, observer freezing, and evaluation remain in the package and staged runner. Restarting the kernel is normally unnecessary for an editable install in a fresh Colab runtime.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{REPO_ROOT}[qwen]"],
    cwd=REPO_ROOT,
    check=True,
)


In [ ]:
# Fail early if the full run is not on a suitable CUDA runtime.
import torch

assert torch.cuda.is_available(), "Select a GPU runtime before the scientific run."
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
approved_gpu = any(label in gpu_name.upper() for label in ("A100", "H100", "H200"))
assert approved_gpu and gpu_memory_gib >= 39, f"The 7B protocol requires an A100/H100/H200 with >=40 GiB; found {gpu_name} ({gpu_memory_gib:.1f} GiB)."
torch.backends.cuda.matmul.allow_tf32 = True
{"gpu": gpu_name, "gpu_memory_gib": round(gpu_memory_gib, 1), "torch": torch.__version__}


## Verify the frozen configuration

Record this SHA-256 digest with the run. A resumed run must use the same digest. The runner also verifies its own stage manifests and checkpoints.


In [ ]:
import json
from pprint import pprint
from observerbench.provenance import json_sha256

config_bytes = CONFIG.read_bytes()
frozen_config = json.loads(config_bytes)
source_manifest = json.loads(SOURCE_MANIFEST.read_text())
CONFIG_SHA256 = json_sha256(frozen_config)
EXPECTED_CONFIG_SHA256 = "ec2835be61a85c6f963cab901c1de17f512e9fa08d5983c79bd14874000bdb77"
assert CONFIG_SHA256 == EXPECTED_CONFIG_SHA256, "Frozen scientific config changed."
assert source_manifest["config_sha256"] == CONFIG_SHA256, "Source seal belongs to another config."
print(f"source_bundle_sha256={source_manifest['source_bundle_sha256']}")
print(f"config_sha256={CONFIG_SHA256}")
pprint(frozen_config, sort_dicts=True)


## Preregistered stage order

1. `prepare`: materialize the six disjoint prompt/token banks.
2. `discover`: run the clean gate, attention scan, and singleton causal discovery on discovery-only data.
3. `confirm`: compare the frozen selected heads with matched controls on held-out confirmation prompts. A failed gate ends the study as a negative result.
4. `freeze-design`: freeze the eight-head Boolean mask design and locked action pools.
5. `measure-calibration`: collect only calibration outcomes.
6. `freeze-observers`: fit and hash observers, predictions, targets, and actions before any locked-test outcome is read.
7. `measure-locked-test`: collect the exact complementary locked outcomes.
8. `evaluate`: score the already-frozen predictions and actions and write the primary final manifests.
9. `measure-collateral`: measure masks selected by the sealed actions on deterministic matched non-induction controls. This secondary diagnostic cannot block or change the primary evaluation.


In [ ]:
# One small wrapper for every stage. `check=True` stops on failed gates.
def run_stage(stage: str, *, config: Path = CONFIG, artifacts_root: Path = ARTIFACTS_ROOT, resume: bool = True):
    command = [
        sys.executable,
        str(SCRIPT),
        "--config",
        str(config),
        "--artifacts-root",
        str(artifacts_root),
        "--stage",
        stage,
    ]
    if resume:
        command.append("--resume")
    print("Running:", " ".join(command))
    return subprocess.run(
        command,
        cwd=REPO_ROOT,
        check=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"},
    )


## 1. Prepare immutable prompt banks

This stage is model-light and safe to rerun. It writes the prompt design and its hashes to the persistent artifact root.


In [ ]:
run_stage("prepare")


## 2. Discover candidate heads

This is the first expensive stage. It uses only reference, discovery, and head-fit banks. If the clean task or causal discovery gate fails, execution stops; do not reinterpret the later stages as scientific evidence.


In [ ]:
run_stage("discover")


## 3. Confirm against matched controls

The selected panel must beat same-layer, same-KV low-induction controls on untouched confirmation prompts. This held-out gate decides whether a second-mechanism claim is available.


In [ ]:
run_stage("confirm")


## 4. Freeze the intervention design

Only a passed confirmation gate can freeze the selected eight-head panel, the 256-mask Boolean cube, the calibration/locked split, and the action pools.


In [ ]:
run_stage("freeze-design")


## 5. Measure calibration outcomes

The runner checkpoints by mask in Drive. If Colab disconnects, rerun this same cell; `--resume` skips verified completed shards.


In [ ]:
run_stage("measure-calibration")


## 6. Freeze observers and actions

This stage fits the no-effect, additive, and quadratic observers at budgets 16, 40, 64, and 128, then commits their predictions and action choices by hash. It must finish before locked outcomes exist.


In [ ]:
run_stage("freeze-observers")


## 7. Measure the locked test

The runner refuses this stage unless the observer/action freeze manifest is complete and valid. As above, mask-level checkpoints make the stage resumable after a preemption.


In [ ]:
run_stage("measure-locked-test")


## 8. Evaluate the frozen primary study

Evaluation may read locked outcomes only after all predictions and actions are frozen. The 128-mask additive-minus-quadratic comparison is primary; smaller budgets are secondary curves. The final report should cover all preregistered targets, their equal-weight aggregate, and the exact no-op action.


In [ ]:
run_stage("evaluate")


## 9. Measure matched-control collateral shift

This secondary stage applies only masks already selected by the sealed actions. It uses deterministic exact-multiset controls and records full-vocabulary KL and total variation. It runs after primary evaluation, so a collateral failure cannot block or change the primary result.


In [ ]:
run_stage("measure-collateral")


## Artifact audit (not interpretation)

This inventory only confirms that the persistent run produced files. Review the final gate report, hashes, confidence intervals, and preregistered metrics before drafting any result or changing the paper.


In [ ]:
artifact_files = sorted(path.relative_to(ARTIFACTS_ROOT).as_posix() for path in ARTIFACTS_ROOT.rglob("*") if path.is_file())
{
    "config_sha256": CONFIG_SHA256,
    "artifact_count": len(artifact_files),
    "last_20_paths": artifact_files[-20:],
}


## Optional engineering smoke test (not scientific evidence)

Use the 0.5B checkpoint only to test model loading, hooks, serialization, and resume behavior. It has a smaller engineering design, writes to a separate directory, and cannot satisfy or substitute for any 7B scientific gate. Set the flag explicitly to run it.


In [ ]:
RUN_ENGINEERING_SMOKE = False
SMOKE_CONFIG = REPO_ROOT / "configs/revision/phase09/qwen2_5_0_5b_induction_smoke.json"
SMOKE_ROOT = ARTIFACTS_ROOT.parent / "engineering_smoke_qwen2_5_0_5b_do_not_cite"

if RUN_ENGINEERING_SMOKE:
    assert SMOKE_CONFIG.is_file(), f"Missing smoke configuration: {SMOKE_CONFIG}"
    SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
    run_stage("engineering-smoke", config=SMOKE_CONFIG, artifacts_root=SMOKE_ROOT)
else:
    print("Engineering smoke disabled. Set RUN_ENGINEERING_SMOKE=True to run it separately.")


## Decision record

After `evaluate` succeeds, record one of three outcomes outside this scaffold:

- **Positive scientific result:** all gates pass and the locked metrics support a preregistered claim. State the exact target, budget, effect size, and uncertainty.
- **Negative scientific result:** a gate fails or the locked comparison does not support the claim. Preserve the artifacts and report the failed criterion directly.
- **Engineering-only result:** only the smoke path completes. Fix the pipeline, but make no claim about Qwen2.5-7B or a second mechanism.
